### **This notebook contains the Python code to conduct a Poisson fixed-effects model on the wildfire and conflict data.**

Task 1: Add administrative data (province, regency, sub-district) to the fire spreadsheet. I choose to use the fire data from the J1-VIIRS C2 satellite from NASA MODIS options. You can download more wildfire datasets from NASA MODIS [here](https://firms.modaps.eosdis.nasa.gov/download/list.php)

In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load data
fires = pd.read_csv("/Users/hariaksha/Documents/GitHub/climate-conflict/data/climate-pressure/fire/DL_FIRE_J1VIIRS-C2_718929_Nov2000-Feb2026_buffer0km-csv/fire_archive_J1V-C2_718929.csv")                     
kecamatan = gpd.read_file("/Users/hariaksha/Documents/GitHub/climate-conflict/data/administrative/batas_kec/batas_kec.shp") 

# Change and inspect column names
kecamatan.rename(columns={"NAME_1": "provinsi", "NAME_2": "kabupaten", "NAME_3": "kecamatan"}, inplace=True)
print("Fires Columns: ", fires.columns.tolist())
# print(provinces.columns.tolist())

# Convert fires CSV → GeoDataFrame of points
fires_gdf = gpd.GeoDataFrame(
    fires,
    geometry=gpd.points_from_xy(fires["longitude"], fires["latitude"]),
    crs="EPSG:4326"
)

# Reproject provinces to match (EPSG:4326 = standard lat/lon)
kecamatan = kecamatan.to_crs("EPSG:4326")

# Spatial join: attach province attributes to each fire point
fires_with_admin = gpd.sjoin(
    fires_gdf,
    kecamatan[["provinsi", "kabupaten", "kecamatan", "geometry"]],  # all three levels
    how="left",
    predicate="within"
)

# Drop geometry and index columns added by sjoin
fires_with_admin = fires_with_admin.drop(columns=["geometry", "index_right"])

# Save result 
fires_with_admin.to_csv("fires_with_admin.csv", index=False)
print("Done! Shape:", fires_with_admin.shape)
print(fires_with_admin[["latitude", "longitude", "provinsi", "kabupaten", "kecamatan"]].head())


Fires Columns:  ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
Done! Shape: (1307603, 18)
   latitude  longitude    provinsi    kabupaten    kecamatan
0  -4.01907  136.07845       Papua      Dogiyai         Kamu
1  -3.85214  136.35695       Papua       Paniai         Kebo
2  -3.98694  136.09912       Papua      Dogiyai   Kamu Utara
3  -4.01018  136.09189       Papua      Dogiyai   Kamu Timur
4  -7.79190  112.01424  Jawa Timur  Kota Kediri  Kota Kediri


Below has concrete Python code to build the panel data and run a Poisson fixed effects model with lagged fires and clustered SEs. In this model, the outcome variable is conflict events, fatalities, or population exposure. The outcome variable can be changed in the last code block where the formula variable is initiated.


Keep in mind that the wellbeing data are province-year and the conflict and fire data are province-week-year.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices  # to build design matrices with FE dummies

# Load wildfire data
fires = pd.read_csv("fires_with_provinces.csv") 

# Load Human Development Index (HDI) data
hdi_raw = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/wellbeing/Human Development Index by Province ((Life Expectancy at Birth SP2020-LF), 2020-25.xlsx") 
hdi_raw.rename(columns={hdi_raw.columns[0]: "provinsi"}, inplace=True)
year_cols = [ # Year selection to avoid value error later
    c for c in hdi_raw.columns
    if isinstance(c, (int, float)) or (isinstance(c, str) and c.strip().isdigit())
]
hdi_sub = hdi_raw[["provinsi"] + year_cols]
hdi_long = (
    hdi_sub
    .melt(id_vars="provinsi", var_name="year", value_name="hdi")
)
hdi_long["year"] = hdi_long["year"].astype(int)
# print(hdi_long.head())

# 4. Load ACLED data
acled = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/unrest/Indonesia_aggregated_data_up_to-2026-02-07.xlsx")

# WEEK is like '28-February-2015' → convert to datetime, then year/week
acled["WEEK"] = pd.to_datetime(acled["WEEK"], format="%d-%B-%Y")
acled["year"] = acled["WEEK"].dt.year
acled["week"] = acled["WEEK"].dt.isocalendar().week.astype(int)

# ADMIN1 is province (e.g. 'Aceh'); normalize to uppercase
acled["provinsi"] = acled["ADMIN1"].str.upper().str.strip()

# Start building province-week fires panel
fires["acq_date"] = pd.to_datetime(fires["acq_date"])
fires["year"] = fires["acq_date"].dt.year
fires["week"] = fires["acq_date"].dt.isocalendar().week.astype(int)
fires["provinsi"] = fires["provinsi"].str.upper().str.strip() # normalize province names to uppercase so they match HDI/ACLED
fires_pw = (
    fires
    .groupby(["provinsi", "year", "week"])
    .agg(
        n_fires=("frp", "size"),
        total_frp=("frp", "sum")
    )
    .reset_index()
)

# Start building province-week conflict panel 
conflict_pw = (
    acled
    .groupby(["provinsi", "year", "week"])
    .agg(
        events=("EVENTS", "sum"),
        fatalities=("FATALITIES", "sum"),
        pop_exposure=("POPULATION_EXPOSURE", "sum")
    )
    .reset_index()
)

# Merge fires, HDI, and conflict
panel = conflict_pw.merge(
    fires_pw,
    on=["provinsi", "year", "week"],
    how="left"
)
panel["n_fires"] = panel["n_fires"].fillna(0)
panel["total_frp"] = panel["total_frp"].fillna(0)
panel = panel.merge( # merge HDI (province–year)
    hdi_long,
    on=["provinsi", "year"],
    how="left"
)
panel = panel.dropna(subset=["hdi"])# optional: drop years without HDI (if outside 2020–2025)

# Add fire lags (autocorrelation)
panel = panel.sort_values(["provinsi", "year", "week"])
panel["n_fires_l1"] = panel.groupby("provinsi")["n_fires"].shift(1)
panel["total_frp_l1"] = panel.groupby("provinsi")["total_frp"].shift(1)
panel[["n_fires_l1", "total_frp_l1"]] = (
    panel[["n_fires_l1", "total_frp_l1"]].fillna(0)
)

# Poisson fixed‑effects regression (province + year‑week FE)
panel["year_week"] = panel["year"] * 100 + panel["week"] # create a combined time FE
formula = "pop_exposure ~ n_fires + n_fires_l1 + hdi + C(provinsi) + C(year_week)" # outcome: number of events per province–week
y, X = dmatrices(formula, panel, return_type="dataframe")
model = sm.GLM(y, X, family=sm.families.Poisson())
res = model.fit(
    cov_type="cluster",
    cov_kwds={"groups": panel["provinsi"]}  # cluster by province
)
print(res.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           pop_exposure   No. Observations:                 1661
Model:                            GLM   Df Residuals:                     1337
Model Family:                 Poisson   Df Model:                          323
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -2.0126e+07
Date:                Mon, 23 Mar 2026   Deviance:                   4.0232e+07
Time:                        11:31:56   Pearson chi2:                 4.42e+07
No. Iterations:                    31   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

I find no statistically significant causal estimates (by p-value) in the last model. This is likely because at the province-week level, the data is too sparse (there are too many zeros). Thus, my current design and model make it hard to find any effect. I list here some ways to adjust the model to find causality.
- Change aggregation from province-week to province-month. This way, the data is less sparse and there are less time FE (so less computationally expensive). Still preserves effects of seasonality and time-specific national shocks.
- Change aggregation from province-week to district-week. I think this will just make it worse because data will be even more sparse, and there will be many more district FE to keep track of then too
- Change outcome from number of events or number of fatalities to probability or binary output to predict occurence of conflict.
- Change outcome to predicting sub-event type (e.g. political violence vs demonstrations) or number of sub-event types.
- Change outcome to predicting number of population exposure to any type of conflict within one week/month/etc
- Keep n_fires but add an intensity/size measure, such as log(1 + total_frp), which is the log of fire radiative power + 1. 

In the following code, I run the same models as before but use a province-month panel instead of a province-week panel. I use fire radiative power (FRP) as a regressor instead of number of fires.

In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from patsy import dmatrices  # to build design matrices with FE dummies

# Load wildfire data
fires = pd.read_csv("fires_with_provinces.csv") 

# Load Human Development Index (HDI) data
hdi_raw = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/wellbeing/Human Development Index by Province ((Life Expectancy at Birth SP2020-LF), 2020-25.xlsx") 
hdi_raw.rename(columns={hdi_raw.columns[0]: "provinsi"}, inplace=True)
year_cols = [ # Year selection to avoid value error later
    c for c in hdi_raw.columns
    if isinstance(c, (int, float)) or (isinstance(c, str) and c.strip().isdigit())
]
hdi_sub = hdi_raw[["provinsi"] + year_cols]
hdi_long = (
    hdi_sub
    .melt(id_vars="provinsi", var_name="year", value_name="hdi")
)
hdi_long["year"] = hdi_long["year"].astype(int)

# 4. Load ACLED data
acled = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/unrest/Indonesia_aggregated_data_up_to-2026-02-07.xlsx")

# WEEK is like '28-February-2015' → convert to datetime, then year/week
acled["WEEK"] = pd.to_datetime(acled["WEEK"], format="%d-%B-%Y")
acled["year"] = acled["WEEK"].dt.year
acled["month"] = acled["WEEK"].dt.month

acled["provinsi"] = acled["ADMIN1"].str.upper().str.strip() # ADMIN1 is province (e.g. 'Aceh'); normalize to uppercase

# Start building province-week fires panel
fires["acq_date"] = pd.to_datetime(fires["acq_date"])
fires["year"] = fires["acq_date"].dt.year
fires["month"] = fires["acq_date"].dt.month
fires["provinsi"] = fires["provinsi"].str.upper().str.strip()
fires_pm = (
    fires
    .groupby(["provinsi", "year", "month"])
    .agg(
        n_fires=("frp", "size"),
        total_frp=("frp", "sum")
    )
    .reset_index()
)
fires_pm["log_total_frp"] = np.log1p(fires_pm["total_frp"]) # fire intensity regressor
fires_pm["log_n_fires"] = np.log1p(fires_pm["n_fires"])  # log count of fire pixels

# Start building province-week conflict panel 
conflict_pm = (
    acled
    .groupby(["provinsi", "year", "month"])
    .agg(
        events=("EVENTS", "sum"),
        fatalities=("FATALITIES", "sum"),
        pop_exposure=("POPULATION_EXPOSURE", "sum")
    )
    .reset_index()
)

# Merge fires, HDI, and conflict at province-month
panel = conflict_pm.merge(
    fires_pm,
    on=["provinsi", "year", "month"],
    how="left"
)
panel["n_fires"] = panel["n_fires"].fillna(0)
panel["total_frp"] = panel["total_frp"].fillna(0)
panel["log_total_frp"] = panel["log_total_frp"].fillna(0)
panel["log_n_fires"] = panel["log_n_fires"].fillna(0)
panel = panel.merge( # merge HDI (province–year)
    hdi_long,
    on=["provinsi", "year"],
    how="left"
)
panel = panel.dropna(subset=["hdi"])
panel = panel.sort_values(["provinsi", "year", "month"]) # add lags of fire intensity by province
panel["log_total_frp_l1"] = panel.groupby("provinsi")["log_total_frp"].shift(1)
panel["log_total_frp_l1"] = panel["log_total_frp_l1"].fillna(0)
panel["log_n_fires_l1"] = panel.groupby("provinsi")["log_n_fires"].shift(1)
panel["log_n_fires_l1"] = panel["log_n_fires_l1"].fillna(0)

# Poisson fixed‑effects regression (province + year‑month FE)
panel["year_month"] = panel["year"] * 100 + panel["month"]
# formula = "events ~ log_total_frp + log_total_frp_l1 + hdi + C(provinsi) + C(year_month)" # outcome: number of events per province–month
# formula = "events ~ log_n_fires + log_n_fires_l1 + hdi + C(provinsi) + C(year_month)" # outcome: number of events per province–month
formula = "events ~ log_n_fires + log_n_fires_l1 + log_total_frp + log_total_frp_l1 + hdi + C(provinsi) + C(year_month)"
y, X = dmatrices(formula, panel, return_type="dataframe")
model = sm.GLM(y, X, family=sm.families.Poisson())
res = model.fit(
    cov_type="cluster",
    cov_kwds={"groups": panel["provinsi"]}
)
print(res.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 events   No. Observations:                  637
Model:                            GLM   Df Residuals:                      551
Model Family:                 Poisson   Df Model:                           85
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1411.2
Date:                Mon, 23 Mar 2026   Deviance:                       741.51
Time:                        11:40:48   Pearson chi2:                     739.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.8310
Covariance Type:              cluster                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

This makes the Poisson FE model on year-week and kabupaten.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import statsmodels.api as sm
from patsy import dmatrices


# Load wildfire data (now with kabupaten)
fires = pd.read_csv("fires_with_admin.csv")


# Load ACLED data
acled = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/unrest/Indonesia_aggregated_data_up_to-2026-02-07.xlsx")
acled["WEEK"] = pd.to_datetime(acled["WEEK"], format="%d-%B-%Y")
acled["year"] = acled["WEEK"].dt.year
acled["week"] = acled["WEEK"].dt.isocalendar().week.astype(int)
acled["provinsi"] = acled["ADMIN1"].str.upper().str.strip()


# Spatially join kabupaten onto ACLED using centroids
kabupaten = gpd.read_file("/Users/hariaksha/Documents/GitHub/climate-conflict/data/administrative/batas_kab/batas_kab.shp")
kabupaten.rename(columns={"NAME_2": "kabupaten"}, inplace=True)
kabupaten = kabupaten.to_crs("EPSG:4326")

acled_gdf = gpd.GeoDataFrame(
    acled,
    geometry=gpd.points_from_xy(acled["CENTROID_LONGITUDE"], acled["CENTROID_LATITUDE"]),
    crs="EPSG:4326"
)
acled_gdf = gpd.sjoin(
    acled_gdf,
    kabupaten[["kabupaten", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right", "geometry"])
acled_gdf["kabupaten"] = acled_gdf["kabupaten"].str.upper().str.strip()


# Load HDI data
hdi_raw = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/wellbeing/Human Development Index by Province ((Life Expectancy at Birth SP2020-LF), 2020-25.xlsx")
hdi_raw.rename(columns={hdi_raw.columns[0]: "provinsi"}, inplace=True)
year_cols = [
    c for c in hdi_raw.columns
    if isinstance(c, (int, float)) or (isinstance(c, str) and c.strip().isdigit())
]
hdi_long = hdi_raw[["provinsi"] + year_cols].melt(id_vars="provinsi", var_name="year", value_name="hdi")
hdi_long["year"] = hdi_long["year"].astype(int)


# Build kabupaten-week fires panel
fires["acq_date"]  = pd.to_datetime(fires["acq_date"])
fires["year"]      = fires["acq_date"].dt.year
fires["week"]      = fires["acq_date"].dt.isocalendar().week.astype(int)
fires["provinsi"]  = fires["provinsi"].str.upper().str.strip()
fires["kabupaten"] = fires["kabupaten"].str.upper().str.strip()

fires_kw = (
    fires
    .groupby(["provinsi", "kabupaten", "year", "week"])
    .agg(n_fires=("frp", "size"), total_frp=("frp", "sum"))
    .reset_index()
)


# Build kabupaten-week conflict panel
conflict_kw = (
    acled_gdf
    .groupby(["provinsi", "kabupaten", "year", "week"])
    .agg(
        events=("EVENTS", "sum"),
        fatalities=("FATALITIES", "sum"),
        pop_exposure=("POPULATION_EXPOSURE", "sum")
    )
    .reset_index()
)


# Merge fires, HDI, and conflict
panel = conflict_kw.merge(
    fires_kw,
    on=["provinsi", "kabupaten", "year", "week"],
    how="left"
)
panel["n_fires"]   = panel["n_fires"].fillna(0)
panel["total_frp"] = panel["total_frp"].fillna(0)
panel = panel.merge(hdi_long, on=["provinsi", "year"], how="left")
panel = panel.dropna(subset=["hdi"])


# Log-transform FRP
panel["log_frp"] = np.log1p(panel["total_frp"])
panel["kabupaten_fe"] = panel["kabupaten"].str.replace(r"[^A-Z0-9]", "_", regex=True)


# Add all lags from week 1 through 16
ALL_LAGS = list(range(1, 17))  # [1, 2, 3, ..., 16]

panel = panel.sort_values(["kabupaten", "year", "week"])
for lag in ALL_LAGS:
    panel[f"log_frp_l{lag}"] = panel.groupby("kabupaten")["log_frp"].shift(lag)

panel[[f"log_frp_l{l}" for l in ALL_LAGS]] = (
    panel[[f"log_frp_l{l}" for l in ALL_LAGS]].fillna(0)
)


# Poisson fixed-effects regression — all 16 lags
panel["year_week"] = panel["year"] * 100 + panel["week"]

lag_terms = " + ".join([f"log_frp_l{l}" for l in ALL_LAGS])
formula = f"events ~ log_frp + {lag_terms} + hdi + C(kabupaten_fe) + C(year_week)"

y, X = dmatrices(formula, panel, return_type="dataframe")
offset = np.log1p(panel.loc[y.index, "pop_exposure"])

model = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset)
res = model.fit(
    cov_type="cluster",
    cov_kwds={"groups": panel.loc[y.index, "kabupaten"]}
)
print(res.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 events   No. Observations:                 1661
Model:                            GLM   Df Residuals:                     1322
Model Family:                 Poisson   Df Model:                          338
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3318.4
Date:                Sat, 28 Mar 2026   Deviance:                       2518.6
Time:                        17:46:36   Pearson chi2:                 6.65e+05
No. Iterations:                    10   Pseudo R-squ. (CS):             0.4699
Covariance Type:              cluster                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

This makes the Poisson FE model on year-week and kecamatan. The results are not statistically significant on this, so ignore it for now

In [16]:
import pandas as pd
import numpy as np
import geopandas as gpd
import statsmodels.api as sm
from patsy import dmatrices


# Load wildfire data (with kecamatan)
fires = pd.read_csv("fires_with_admin.csv")


# Load ACLED data
acled = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/unrest/Indonesia_aggregated_data_up_to-2026-02-07.xlsx")
acled["WEEK"] = pd.to_datetime(acled["WEEK"], format="%d-%B-%Y")
acled["year"] = acled["WEEK"].dt.year
acled["week"] = acled["WEEK"].dt.isocalendar().week.astype(int)
acled["provinsi"] = acled["ADMIN1"].str.upper().str.strip()


# Spatially join kecamatan onto ACLED using centroids
kecamatan = gpd.read_file("/Users/hariaksha/Documents/GitHub/climate-conflict/data/administrative/batas_kec/batas_kec.shp")
kecamatan.rename(columns={"NAME_3": "kecamatan"}, inplace=True)  # adjust if needed
kecamatan = kecamatan.to_crs("EPSG:4326")

acled_gdf = gpd.GeoDataFrame(
    acled,
    geometry=gpd.points_from_xy(acled["CENTROID_LONGITUDE"], acled["CENTROID_LATITUDE"]),
    crs="EPSG:4326"
)
acled_gdf = gpd.sjoin(
    acled_gdf,
    kecamatan[["kecamatan", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right", "geometry"])
acled_gdf["kecamatan"] = acled_gdf["kecamatan"].str.upper().str.strip()


# Load HDI data
hdi_raw = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/wellbeing/Human Development Index by Province ((Life Expectancy at Birth SP2020-LF), 2020-25.xlsx")
hdi_raw.rename(columns={hdi_raw.columns[0]: "provinsi"}, inplace=True)
year_cols = [
    c for c in hdi_raw.columns
    if isinstance(c, (int, float)) or (isinstance(c, str) and c.strip().isdigit())
]
hdi_long = hdi_raw[["provinsi"] + year_cols].melt(id_vars="provinsi", var_name="year", value_name="hdi")
hdi_long["year"] = hdi_long["year"].astype(int)


# Build kecamatan-week fires panel
fires["acq_date"]   = pd.to_datetime(fires["acq_date"])
fires["year"]       = fires["acq_date"].dt.year
fires["week"]       = fires["acq_date"].dt.isocalendar().week.astype(int)
fires["provinsi"]   = fires["provinsi"].str.upper().str.strip()
fires["kecamatan"]  = fires["kecamatan"].str.upper().str.strip()

fires_kw = (
    fires
    .groupby(["provinsi", "kecamatan", "year", "week"])
    .agg(n_fires=("frp", "size"), total_frp=("frp", "sum"))
    .reset_index()
)


# Build kecamatan-week conflict panel
conflict_kw = (
    acled_gdf
    .groupby(["provinsi", "kecamatan", "year", "week"])
    .agg(
        events=("EVENTS", "sum"),
        fatalities=("FATALITIES", "sum"),
        pop_exposure=("POPULATION_EXPOSURE", "sum")
    )
    .reset_index()
)


# Merge fires, HDI, and conflict
panel = conflict_kw.merge(
    fires_kw,
    on=["provinsi", "kecamatan", "year", "week"],
    how="left"
)
panel["n_fires"]   = panel["n_fires"].fillna(0)
panel["total_frp"] = panel["total_frp"].fillna(0)
panel = panel.merge(hdi_long, on=["provinsi", "year"], how="left")
panel = panel.dropna(subset=["hdi"])


# Log-transform FRP
panel["log_frp"] = np.log1p(panel["total_frp"])

# Sanitize kecamatan names for patsy
panel["kecamatan_fe"] = panel["kecamatan"].str.replace(r"[^A-Z0-9]", "_", regex=True)


# Add all lags from week 1 through 16
ALL_LAGS = list(range(1, 17))

panel = panel.sort_values(["kecamatan", "year", "week"])
for lag in ALL_LAGS:
    panel[f"log_frp_l{lag}"] = panel.groupby("kecamatan")["log_frp"].shift(lag)

panel[[f"log_frp_l{l}" for l in ALL_LAGS]] = (
    panel[[f"log_frp_l{l}" for l in ALL_LAGS]].fillna(0)
)


# Poisson fixed-effects regression — kecamatan + year-week FE
panel["year_week"] = panel["year"] * 100 + panel["week"]

lag_terms = " + ".join([f"log_frp_l{l}" for l in ALL_LAGS])
formula = f"events ~ log_frp + {lag_terms} + hdi + C(kecamatan_fe) + C(year_week)"

y, X = dmatrices(formula, panel, return_type="dataframe")
offset = np.log1p(panel.loc[y.index, "pop_exposure"])

model = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset)
res = model.fit(
    cov_type="cluster",
    cov_kwds={"groups": panel.loc[y.index, "kecamatan"]}  # cluster by kecamatan
)
print(res.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 events   No. Observations:                 1661
Model:                            GLM   Df Residuals:                     1322
Model Family:                 Poisson   Df Model:                          338
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3327.4
Date:                Sat, 28 Mar 2026   Deviance:                       2536.7
Time:                        17:54:07   Pearson chi2:                 6.96e+05
No. Iterations:                    10   Pseudo R-squ. (CS):             0.4641
Covariance Type:              cluster                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

This next code does the same model as above except on pixels of equal size.

In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import statsmodels.api as sm
from patsy import dmatrices
from shapely.geometry import box

# Load wildfire data
fires = pd.read_csv("fires_with_admin.csv")

# Load ACLED data
acled = pd.read_excel("/Users/hariaksha/Documents/GitHub/climate-conflict/data/unrest/Indonesia_aggregated_data_up_to-2026-02-07.xlsx")
acled["WEEK"] = pd.to_datetime(acled["WEEK"], format="%d-%B-%Y")
acled["year"] = acled["WEEK"].dt.year
acled["week"] = acled["WEEK"].dt.isocalendar().week.astype(int)

# --- Build uniform pixel grid over Indonesia ---
# 0.5 degree resolution (~55km x 55km at equator) — adjust as needed
# Indonesia bounding box: lon 95–141, lat -11–6
RESOLUTION = 0.5

lon_bins = np.arange(95, 141, RESOLUTION)
lat_bins = np.arange(-11, 6, RESOLUTION)

grid_cells = []
cell_ids = []
for i, lon in enumerate(lon_bins):
    for j, lat in enumerate(lat_bins):
        cell_ids.append(f"CELL_{i:03d}_{j:03d}")
        grid_cells.append(box(lon, lat, lon + RESOLUTION, lat + RESOLUTION))

grid = gpd.GeoDataFrame({"cell_id": cell_ids, "geometry": grid_cells}, crs="EPSG:4326")


# --- Assign fires to grid cells ---
fires["acq_date"] = pd.to_datetime(fires["acq_date"])
fires["year"]     = fires["acq_date"].dt.year
fires["week"]     = fires["acq_date"].dt.isocalendar().week.astype(int)

fires_gdf = gpd.GeoDataFrame(
    fires,
    geometry=gpd.points_from_xy(fires["longitude"], fires["latitude"]),
    crs="EPSG:4326"
)
fires_gdf = gpd.sjoin(
    fires_gdf,
    grid[["cell_id", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right", "geometry"])

fires_grid = (
    fires_gdf
    .groupby(["cell_id", "year", "week"])
    .agg(n_fires=("frp", "size"), total_frp=("frp", "sum"))
    .reset_index()
)


# --- Assign ACLED events to grid cells ---
acled_gdf = gpd.GeoDataFrame(
    acled,
    geometry=gpd.points_from_xy(acled["CENTROID_LONGITUDE"], acled["CENTROID_LATITUDE"]),
    crs="EPSG:4326"
)
acled_gdf = gpd.sjoin(
    acled_gdf,
    grid[["cell_id", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right", "geometry"])

conflict_grid = (
    acled_gdf
    .groupby(["cell_id", "year", "week"])
    .agg(
        events=("EVENTS", "sum"),
        fatalities=("FATALITIES", "sum"),
        pop_exposure=("POPULATION_EXPOSURE", "sum")
    )
    .reset_index()
)

# --- Merge fires and conflict with OUTER join to keep all active cells ---
panel = conflict_grid.merge(
    fires_grid,
    on=["cell_id", "year", "week"],
    how="outer"          # ← changed from "left" to "outer"
)
panel["n_fires"]   = panel["n_fires"].fillna(0)
panel["total_frp"] = panel["total_frp"].fillna(0)
panel["events"]    = panel["events"].fillna(0)
panel["fatalities"]    = panel["fatalities"].fillna(0)
panel["pop_exposure"]  = panel["pop_exposure"].fillna(0)
panel = panel.dropna(subset=["cell_id"]) # Drop rows where cell_id is null (points outside Indonesia bounding box)

# Filter to cells with at least one conflict event ever recorded
active_conflict_cells = conflict_grid["cell_id"].unique()
panel = panel[panel["cell_id"].isin(active_conflict_cells)]

# After filtering
# print("Cells in filtered panel:", panel["cell_id"].nunique())
# print("Obs in filtered panel:  ", len(panel))
# # Check how many of those 38 cells also have fire events
# has_fire = panel[panel["total_frp"] > 0]["cell_id"].nunique()
# print("Conflict cells with any fire:", has_fire)

# # --- Diagnostic: check cell counts before fitting ---
# print("Cells with fires:   ", fires_grid["cell_id"].nunique())
# print("Cells with conflict:", conflict_grid["cell_id"].nunique())
# print("Cells in panel:     ", panel["cell_id"].nunique())
# print("Obs in panel:       ", len(panel))

# --- Log-transform FRP and build lags ---
panel["log_frp"] = np.log1p(panel["total_frp"])
panel["cell_fe"] = panel["cell_id"].str.replace(r"[^A-Z0-9]", "_", regex=True)

ALL_LAGS = list(range(1, 17))

panel = panel.sort_values(["cell_id", "year", "week"])
for lag in ALL_LAGS:
    panel[f"log_frp_l{lag}"] = panel.groupby("cell_id")["log_frp"].shift(lag)

panel[[f"log_frp_l{l}" for l in ALL_LAGS]] = (
    panel[[f"log_frp_l{l}" for l in ALL_LAGS]].fillna(0)
)


# --- Poisson FE regression: cell + year-week FE, no HDI ---
panel["year_week"] = panel["year"] * 100 + panel["week"]

lag_terms = " + ".join([f"log_frp_l{l}" for l in ALL_LAGS])
formula = f"events ~ log_frp + {lag_terms} + C(cell_fe) + C(year_week)"

y, X = dmatrices(formula, panel, return_type="dataframe")
offset = np.log1p(panel.loc[y.index, "pop_exposure"])

model = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset)
res = model.fit(
    cov_type="cluster",
    cov_kwds={"groups": panel.loc[y.index, "cell_id"]}
)
print(res.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 events   No. Observations:                12273
Model:                            GLM   Df Residuals:                    11641
Model Family:                 Poisson   Df Model:                          631
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -23739.
Date:                Sat, 28 Mar 2026   Deviance:                       22853.
Time:                        22:37:09   Pearson chi2:                 7.55e+06
No. Iterations:                    24   Pseudo R-squ. (CS):             0.6152
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           